# Step 5 — 两阶段 ZIP-RC predictor 训练

1. Stage 1：用 `correct`、KL=0 训练 intermediate predictor；
2. 用 intermediate model 给 train split 写入 denoised `value`；
3. Stage 2：用 `value`、KL=10 训练最终 ZIP-RC。

训练脚本会把每个 optimizer step 的 loss 写到 JSONL，本 Notebook 绘制 loss、KL 和学习率曲线。

In [ ]:
# @title Step 05.1 — 初始化运行环境
from pathlib import Path
import gc
import importlib.util
import json
import os
import shutil
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
ZIP_PY = Path("/content/mamba/envs/zip/bin/python")
REPO_URL = "https://github.com/wtree101/ZIP-RC-Colab.git"
REPO_BRANCH = "main"
SYNC_REPO = True

if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO)], check=True)
elif SYNC_REPO:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

if not ZIP_PY.exists():
    raise FileNotFoundError(
        f"未找到 {ZIP_PY}。请先建立 ZIP-RC 的 mamba 环境，再重新运行本 Notebook。"
    )

kernel_required = ["torch", "numpy", "pandas", "pyarrow", "matplotlib", "sklearn", "psutil"]
kernel_missing = [name for name in kernel_required if importlib.util.find_spec(name) is None]
if kernel_missing:
    raise ModuleNotFoundError(f"Colab kernel 缺少可视化依赖: {kernel_missing}")

env_check = subprocess.run(
    [
        str(ZIP_PY),
        "-c",
        (
            "import importlib.util, json; "
            "mods=['torch','vllm','transformers','datasets','pandas','pyarrow']; "
            "print(json.dumps([m for m in mods if importlib.util.find_spec(m) is None]))"
        ),
    ],
    check=True,
    capture_output=True,
    text=True,
)
env_missing = json.loads(env_check.stdout.strip())
if env_missing:
    raise ModuleNotFoundError(f"zip mamba 环境缺少依赖: {env_missing}")

os.environ["ZIPRC_PYTHON"] = str(ZIP_PY)

import matplotlib.pyplot as plt
import pandas as pd
import psutil
import torch
from IPython.display import display

sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import (
    gate,
    gate_frame,
    load_config,
    model_artifacts_exist,
    read_jsonl,
    require_columns,
    rolling_edges,
    run_repo,
    save_stage_report,
)

print("Repository:", REPO)
print("ZIP Python:", ZIP_PY)

CONFIG = load_config(REPO)
print("Experiment:", CONFIG["experiment_name"])


In [ ]:
# @title Step 05.2 — Stage 1：训练 correctness head
RUN_STAGE_1 = True
train_path = REPO / CONFIG["paths"]["train"]
intermediate_model = REPO / CONFIG["paths"]["intermediate_model"]
stage1_metrics = REPO / CONFIG["paths"]["stage1_metrics"]
if RUN_STAGE_1:
    run_repo(
        REPO,
        "python3", "src/train_ziprc_joint_head.py",
        "--model-id", CONFIG["model_id"], "--data-path", train_path,
        "--weights-path", intermediate_model,
        "--distribution-token-id", CONFIG["distribution_token_id"],
        "--label-column", "correct", "--reward-values", 0.0, 1.0,
        "--kl-coefficient", 0.0,
        "--batch-size", CONFIG["batch_size"],
        "--gradient-accumulation-steps", CONFIG["gradient_accumulation_steps"],
        "--num-epochs", CONFIG["num_epochs"], "--max-steps", CONFIG["stage1_steps"],
        "--learning-rate", CONFIG["stage1_learning_rate"],
        "--max-length", CONFIG["train_max_length"], "--dtype", CONFIG["dtype"],
        "--metrics-path", stage1_metrics, "--log-every", 10, "--visualization-freq", 50,
    )

In [ ]:
# @title Step 05.3 — 生成中间 value
RUN_VALUE_SCORING = True
train_value_path = REPO / CONFIG["paths"]["train_value"]
if RUN_VALUE_SCORING:
    run_repo(
        REPO,
        "python3", "src/score_with_ziprc_joint_head.py",
        "--model", intermediate_model, "--in-parquet", train_path,
        "--out-parquet", train_value_path,
        "--distribution-token-id", CONFIG["distribution_token_id"],
        "--num-length-bins", CONFIG["num_length_bins"],
        "--reward-values", 0.0, 1.0, "--last-k", 64,
        "--max-length", CONFIG["train_max_length"], "--batch-size", 1,
        "--num-workers", 0, "--dtype", CONFIG["dtype"],
    )

In [ ]:
# @title Step 05.4 — Stage 2：训练最终 predictor
RUN_STAGE_2 = True
final_model = REPO / CONFIG["paths"]["final_model"]
stage2_metrics = REPO / CONFIG["paths"]["stage2_metrics"]
if RUN_STAGE_2:
    run_repo(
        REPO,
        "python3", "src/train_ziprc_joint_head.py",
        "--model-id", CONFIG["model_id"], "--data-path", train_value_path,
        "--weights-path", final_model,
        "--distribution-token-id", CONFIG["distribution_token_id"],
        "--label-column", "value", "--reward-values", *CONFIG["reward_values"],
        "--kl-coefficient", 10.0,
        "--batch-size", CONFIG["batch_size"],
        "--gradient-accumulation-steps", CONFIG["gradient_accumulation_steps"],
        "--num-epochs", CONFIG["num_epochs"], "--max-steps", CONFIG["stage2_steps"],
        "--learning-rate", CONFIG["stage2_learning_rate"],
        "--max-length", CONFIG["train_max_length"], "--dtype", CONFIG["dtype"],
        "--metrics-path", stage2_metrics, "--log-every", 10, "--visualization-freq", 50,
    )

In [ ]:
# @title Step 05.5 — 检查 loss 与 value 分离
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

histories = {"stage1": read_jsonl(stage1_metrics), "stage2": read_jsonl(stage2_metrics)}
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for row, (name, history) in enumerate(histories.items()):
    smooth = history.set_index("step")[["total_loss", "distribution_loss", "kl_loss"]].rolling(15, min_periods=1).mean()
    smooth.plot(ax=axes[row, 0])
    axes[row, 0].set_title(f"{name}: smoothed losses")
    history.plot(x="step", y="learning_rate", ax=axes[row, 1], legend=False, color="#f58518")
    axes[row, 1].set_title(f"{name}: learning rate")
plt.tight_layout()
plt.show()

stage1_start, stage1_end = rolling_edges(histories["stage1"]["distribution_loss"])
stage2_start, stage2_end = rolling_edges(histories["stage2"]["total_loss"])
value_df = pd.read_parquet(train_value_path)
value_std = float(value_df["value"].std())

fig, ax = plt.subplots(figsize=(7, 4))
for label, group in value_df.groupby("correct"):
    ax.hist(group["value"].dropna(), bins=30, alpha=.55, density=True, label=f"correct={label}")
ax.set(title="Intermediate value separation on train", xlabel="predicted value")
ax.legend()
plt.show()

checks = [
    gate("Stage 1 模型已保存", model_artifacts_exist(intermediate_model), str(intermediate_model)),
    gate("Stage 2 模型已保存", model_artifacts_exist(final_model), str(final_model)),
    gate("训练 metrics 完整", len(histories["stage1"]) >= 50 and len(histories["stage2"]) >= 50, f"steps={len(histories['stage1'])}/{len(histories['stage2'])}"),
    gate("Loss 全部有限", all(np.isfinite(history[["total_loss", "distribution_loss", "kl_loss"]]).all().all() for history in histories.values()), "no NaN/Inf"),
    gate("Stage 1 distribution loss 下降", stage1_end < stage1_start, f"{stage1_start:.3f} → {stage1_end:.3f}", kind="scientific"),
    gate("Stage 2 total loss 下降", stage2_end < stage2_start, f"{stage2_start:.3f} → {stage2_end:.3f}", kind="scientific"),
    gate("Value 非常数", value_std >= .01, f"std={value_std:.4f}", kind="scientific"),
]
display(gate_frame(checks))
save_stage_report(REPO, "05_predictor_training", checks, {"stage1_loss_start": stage1_start, "stage1_loss_end": stage1_end, "stage2_loss_start": stage2_start, "stage2_loss_end": stage2_end, "value_std": value_std})